# Import \+ Load dataset 

In [ ]:
%pip install -q kagglehub pandas numpy scipy matplotlib seaborn scikit-learn statsmodels xgboost catboost  openpyxl

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels as sm

from pathlib import Path
from scipy.fft import fft, fftfreq

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score,root_mean_squared_error
)
from sklearn.feature_selection import SelectKBest, f_regression, RFE, mutual_info_regression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from catboost import CatBoostRegressor

from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.statespace.sarimax import SARIMAX
# Replace with the Kaggle dataset's owner slug
path = kagglehub.dataset_download("vishala28/swat-dataset-secure-water-treatment-system")

print("Path to dataset files:", path)

In [ ]:
df = pd.read_csv(Path(path) / "merged.csv")

# EDA of dataset

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes.to_string())
print("\nMissing values:")
print(df.isna().sum()[df.isna().sum() > 0])
print("\nDuplicates:", df.duplicated().sum())
print("\nFirst 5 rows:")
display(df.head())
print("\nLast 5 rows:")
display(df.tail())

# Preprocess data

## Handling timestamp 

In [ ]:
data = df.copy()

# remove leading/trailing spaces from column names
data.columns = data.columns.str.strip()

# parse timestamps (day/month/year, 12-hour clock)
data["Timestamp"] = pd.to_datetime(
    data["Timestamp"].astype(str).str.strip(),
    format="%d/%m/%Y %I:%M:%S %p",
    errors="raise"
)

# remove spaces from the Normal/Attack labels
data["Normal/Attack"] = data["Normal/Attack"].astype(str).str.strip()

# the merged file in this dataset repeats the normal-operation records, so drop exact duplicate rows
rows_before = len(data)
data = data.drop_duplicates()
print(f"Exact duplicate rows removed: {rows_before - len(data):,} ({rows_before:,} -> {len(data):,})")

# sort by time and use the timestamp as the index
data = data.sort_values("Timestamp").set_index("Timestamp")

print("Clean shape:", data.shape)
print("Invalid timestamps:", data.index.isna().sum())
print("Duplicate timestamps:", data.index.duplicated().sum())
print("Chronological:", data.index.is_monotonic_increasing)

display(data["LIT101"].describe())

if not data.index.is_unique:
    raise ValueError('Conflicting duplicate timestamps remain after removing exact duplicates.')
if not data['Normal/Attack'].isin(['Normal', 'Attack']).all():
    raise ValueError('Unknown or missing operation labels; inspect them before resampling.')

# Warm-up cutoff documented by the plant start-up plots below; freeze before modelling.
WARMUP_END = pd.Timestamp('2015-12-23 15:00:00')
attack_times = data.index[data['Normal/Attack'].eq('Attack')]

first_attack = attack_times.min().floor('min')
normal_minutes = pd.date_range(max(WARMUP_END, data.index.min().ceil('min')),
                              first_attack - pd.Timedelta(minutes=1), freq='min')
development_size = int(len(normal_minutes) * 0.8)
TEST_START = normal_minutes[development_size]
VAL_START = normal_minutes[int(development_size * 0.8)]


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 8))

data["LIT101"].plot(
    ax=axes[0],
    linewidth=0.5,
    title="SWaT: LIT101 Water Level"
)

data.loc["2015-12-31", "LIT101"].plot(
    ax=axes[1],
    linewidth=0.8,
    title="LIT101: December 31, 2015"
)

for ax in axes:
    ax.set_ylabel("Water Level")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
processed = data.copy()
sensor_cols = processed.columns.drop('Normal/Attack') # other columns are sensors, actuators or timestamp
# Learn column availability and variability from the fit period only.
fit_raw = processed.loc[(processed.index >= WARMUP_END) & (processed.index < VAL_START)]
required_sensors = ['LIT101', 'FIT101', 'FIT201']
constant_cols = [c for c in sensor_cols if fit_raw[c].nunique(dropna=True) == 1
                 and c not in required_sensors]
print('Constant in train period, removed:', constant_cols)
processed = processed.drop(columns=constant_cols)


In [ ]:
processed.columns

In [ ]:
print(processed.isna().sum()[lambda x: x > 0])

In [ ]:
missing_summary = pd.DataFrame({
    'fit_missing_count': fit_raw[processed.columns].isna().sum(),
    'fit_missing_pct': fit_raw[processed.columns].isna().mean() * 100,
})
display(missing_summary.loc[missing_summary.fit_missing_count > 0])
# Entirely absent channels cannot support training. Partial gaps are not grounds
# for deleting a sensor: retain them for the resampling/gap checks below.
missing_cols = [c for c in sensor_cols if c in processed and fit_raw[c].isna().all()]
print('Absent throughout fit period, removed:', missing_cols)
processed = processed.drop(columns=missing_cols)
print('Remaining missing values:', processed.isna().sum()[lambda x: x > 0])


## Resample

In [ ]:
numeric_cols = data.select_dtypes(include="number")

unique_counts = numeric_cols.nunique().sort_values()

for col, count in unique_counts.items():
    values = sorted(numeric_cols[col].dropna().unique())

    print(f"\n{col} | {count} unique values")
    print("Values:", values if count <= 10 else values[:10])

In [ ]:
numeric_cols = processed.select_dtypes(include="number")

unique_counts = numeric_cols.nunique().sort_values()

for col, count in unique_counts.items():
    values = sorted(numeric_cols[col].dropna().unique())

    print(f"\n{col} | {count} unique values")
    print("Values:", values if count <= 10 else values[:10])

### What this means

discrete vals <=3 unique vals\. Else continuous ,  Categorical values are not represented here <\- Old insight, disregarded now

In [ ]:
numeric = processed.select_dtypes(include='number')
import re
discrete_cols = [c for c in numeric if re.fullmatch(r'(P|MV|UV)\d+', c)]
continuous_cols = [c for c in numeric if c not in discrete_cols]
print('Actuator states:', discrete_cols)
print('Continuous measurements:', continuous_cols)


In [ ]:
minute_counts = processed["LIT101"].resample("1min").count()

print("Empty minutes:", (minute_counts == 0).sum())
print("Partial minutes:", minute_counts.between(1, 59).sum())

display(minute_counts[minute_counts < 60])

In [ ]:
processed.loc[
    "2015-12-31 20:57":"2015-12-31 21:04",
    ["Normal/Attack", "LIT101"]
]

In [ ]:
# one row per minute: sensors averaged, pump/valve states take their last value in the minute
aggregation = {
    **{c: "mean" for c in continuous_cols},
    **{c: "last" for c in discrete_cols}
}
processed_1min = processed.resample("1min").agg(aggregation)

print("Missing values after resampling:", int(processed_1min.isna().sum().sum()))

# number of raw one-second readings behind each minute
processed_1min["sample_count"] = processed["LIT101"].resample("1min").count()

# a minute is labelled Attack if any second inside it was an attack
processed_1min["Normal/Attack"] = (
    processed["Normal/Attack"]
    .resample("1min")
    .agg(lambda x: "Attack" if x.eq("Attack").any() else "Normal" if len(x) else None)
)

display(processed_1min.head())

### Outlier Handling

In [ ]:
z = (processed_1min[continuous_cols] - processed_1min[continuous_cols].mean()) / processed_1min[continuous_cols].std()
outlier_mask = np.abs(z) > 3

outlier_mask.sum().sort_values(ascending=False)

In [ ]:
WARMUP_END = pd.Timestamp("2015-12-23 15:00:00")

fig, axes = plt.subplots(2, 1, figsize=(15, 6), sharex=True)
for ax, col in zip(axes, ["AIT401", "LIT101"]):
    processed_1min.loc[:"2015-12-24 00:00", col].plot(ax=ax, linewidth=0.7)
    ax.axvline(WARMUP_END, color="red", linestyle="--", label="end of warm-up")
    ax.set_ylabel(col); ax.grid(True, alpha=0.3); ax.legend()
axes[0].set_title("Plant start-up: AIT401 offline and LIT101 filling from empty before the cutoff")
plt.tight_layout(); plt.show()

In [ ]:
is_attack = processed_1min["Normal/Attack"] == "Attack"
#Warm-up = everything before the plant reaches steady state
is_warmup = processed_1min.index < WARMUP_END

summary = pd.DataFrame({
    "outliers": outlier_mask.sum(),
    "warm-up": outlier_mask.apply(lambda c: (c & is_warmup).sum()),
    "attack": outlier_mask.apply(lambda c: (c & is_attack).sum()),
})
summary["other"] = summary["outliers"] - summary["warm-up"] - summary["attack"]
summary.sort_values("outliers", ascending=False)

In [ ]:
processed_clean = processed_1min[~is_warmup].copy()

print(processed_1min.shape, "->", processed_clean.shape)
print("Attack rows kept:", (processed_clean["Normal/Attack"] == "Attack").sum())

In [ ]:
# outlier check against normal operation only: after warm-up, before the first attack
first_attack = processed_clean.index[processed_clean["Normal/Attack"].eq("Attack")].min()
normal_check = processed_clean.loc[processed_clean.index < first_attack, continuous_cols]

# z-score: how many standard deviations each reading is from its column mean
z_normal = (normal_check - normal_check.mean()) / normal_check.std()
outliers_normal = (np.abs(z_normal) > 3).sum().sort_values(ascending=False)

print("Outliers (|z| > 3) during normal operation:")
print(outliers_normal[outliers_normal > 0])
print("\nLIT101 outliers:", int(outliers_normal.get("LIT101", 0)))

In [ ]:
# inspect every flagged sensor: is each outlier an isolated spike or part of a sustained, plausible pattern?
flagged = outliers_normal[outliers_normal > 0].index.tolist()

fig, axes = plt.subplots(len(flagged), 1, figsize=(15, 3 * len(flagged)), sharex=True)
axes = np.atleast_1d(axes)
for ax, col in zip(axes, flagged):
    series = normal_check[col]
    is_out = np.abs(z_normal[col]) > 3
    ax.plot(series.index, series, linewidth=0.6, label=col)
    ax.scatter(series.index[is_out], series[is_out], color="red", s=10, label="|z| > 3")
    ax.set_ylabel(col); ax.legend(loc="upper right"); ax.grid(True, alpha=0.3)
axes[0].set_title("Flagged sensors during normal operation (red = flagged readings)")
plt.tight_layout(); plt.show()

# how the flagged readings are grouped: many short runs suggest spikes, few long runs suggest real events
for col in flagged:
    is_out = (np.abs(z_normal[col]) > 3).astype(int)
    runs = (is_out.diff() == 1).sum() + int(is_out.iloc[0] == 1)   # number of separate flagged stretches
    print(f"{col}: {int(is_out.sum())} flagged minutes in {runs} separate stretches | "
          f"std = {normal_check[col].std():.4f} | min = {normal_check[col].min():.4f} | max = {normal_check[col].max():.4f}")

During normal operation, LIT101 has no outliers (|z| > 3). Three other sensors are flagged, and inspection shows none are sensor faults:

AIT401 varies by only 0.024 units in total (std 0.003), so single resolution steps of the instrument exceed |z| = 3; the z-score is not meaningful for a nearly constant signal.
FIT601 is a periodically operated pump: flow is zero except for regular one-minute pulses of similar height, all flagged because the mean is near zero.
PIT502 shows a smooth, sustained pressure rise over the final hours, a change in operating state reaching levels similar to those seen earlier in the period.

All values are therefore kept. This also shows the limitation of the z-score method, which assumes a roughly normal distribution and misfires on near-constant, mostly-zero or slowly shifting signals.

In [ ]:
target = 'LIT101'
processed_clean.to_csv('swat_post_processed.csv')
normal_1min = processed_clean.loc[processed_clean.index < first_attack].asfreq('min').copy()
if not normal_1min.index.equals(normal_minutes):
    raise ValueError('Normal-operation minute grid differs from the declared split grid.')
if normal_1min[required_sensors].isna().any().any():
    raise ValueError('Required sensor gaps need investigation; do not bridge them with future readings.')
if not normal_1min['sample_count'].eq(60).all():
    raise ValueError('Incomplete minutes in the modelling period need explicit treatment.')
ml_df = normal_1min[[target]].copy()
train = ml_df.loc[ml_df.index < TEST_START].copy()
test = ml_df.loc[ml_df.index >= TEST_START].copy()
print('Fit:', train.index.min(), 'to', train.index[train.index < VAL_START].max())
print('Validation:', VAL_START, 'to', train.index.max())
print('Test:', TEST_START, 'to', test.index.max())

# FFT identifies a cycle from fit data only; no FFT amplitudes become ML features.
s = train.loc[train.index < VAL_START, target]
frequencies = fftfreq(len(s), d=1)
magnitudes = np.abs(fft((s - s.mean()).to_numpy()))
band = (frequencies > 1/360) & (frequencies < 1/5) # less than 5 minutes and more than 6 hours are not allowed
CYCLE = int(round(1 / frequencies[band][np.argmax(magnitudes[band])]))
print('Dominant fit-period cycle:', CYCLE, 'minutes')
plt.figure(figsize=(10, 4))
plt.plot(1/frequencies[band], magnitudes[band])
plt.axvline(CYCLE, color='red', linestyle='--', label=f'{CYCLE} minutes')
plt.xlabel('Period (minutes)'); plt.ylabel('FFT magnitude'); plt.title('Fit-period cycle evidence')
plt.legend(); plt.tight_layout(); plt.show()

def regression_metrics(actual, predicted):
 
    actual = np.asarray(actual, dtype=float).reshape(-1)
    predicted = np.asarray(predicted, dtype=float).reshape(-1)
    
    mse = np.mean((actual-predicted)**2)
    # MAPE is undefined if any actual is zero; do not silently omit those rows.
    mape = np.nan if np.any(np.abs(actual) < 1e-8) else np.mean(np.abs((actual-predicted)/actual))*100
    return {'MAE': np.mean(np.abs(actual-predicted)), 'MSE': mse,
            'RMSE': np.sqrt(mse), 'MAPE (%)': mape}


# Feature engineering and extraction for ML

In [ ]:
def create_all_features(data, cycle=CYCLE, drop_incomplete=True):
    df_feat = data.copy()

    # calendar features: hour of day, and hour as a cycle (sin/cos)
    df_feat["hour"] = df_feat.index.hour
    df_feat["sin_hour"] = np.sin(2 * np.pi * df_feat.index.hour / 24)
    df_feat["cos_hour"] = np.cos(2 * np.pi * df_feat.index.hour / 24)

    # trend features: position in time (counted from the start of the series) and its square
    df_feat["trend"] = ((df_feat.index - ml_df.index[0]) / pd.Timedelta(minutes=1)).to_numpy()
    df_feat["trend_squared"] = df_feat["trend"] ** 2

    # lag features: recent minutes plus one full fill/drain cycle ago
    for lag in sorted({1, 2, 3, 6, 12, 24, cycle}):
        df_feat[f"lag_{lag}"] = df_feat[target].shift(lag)

    # rolling mean and std of the previous window (shift(1) excludes the current value)
    for window in [3, 6, 12]:
        df_feat[f"rolling_mean_{window}"] = df_feat[target].shift(1).rolling(window).mean()
        df_feat[f"rolling_std_{window}"] = df_feat[target].shift(1).rolling(window).std()

    # ratio: last value relative to one cycle ago
    df_feat[f"ratio_lag1_lag{cycle}"] = df_feat["lag_1"] / df_feat[f"lag_{cycle}"].replace(0, np.nan)


    feature_cols = df_feat.columns.drop(target)
    df_feat[feature_cols] = df_feat[feature_cols].replace([np.inf, -np.inf], np.nan)
    return df_feat.dropna(subset=feature_cols) if drop_incomplete else df_feat

# Each feature at t uses only values before t. Carry training history into the
# test boundary; dropping the first CYCLE test rows would discard valid forecasts.
features = create_all_features(ml_df)
train_feat = features.loc[features.index < TEST_START]
test_feat = features.loc[features.index >= TEST_START]
X_train_full, y_train = train_feat.drop(columns=target), train_feat[target]
X_test_full, y_test = test_feat.drop(columns=target), test_feat[target]
if not y_test.index.equals(test.index):
    raise ValueError('Feature construction unexpectedly removed test minutes.')
split = int((X_train_full.index < VAL_START).sum())
X_fit_full, X_val_full = X_train_full.iloc[:split], X_train_full.iloc[split:]
y_train_final, y_val_final = y_train.iloc[:split], y_train.iloc[split:]
y_test_final = y_test
print('Features:', X_train_full.shape[1], '| fit / validation / test:',
      len(X_fit_full), len(X_val_full), len(X_test_full))


In [ ]:
# score each feature with a univariate F-test against the target, keep the best 8
filter_selector = SelectKBest(score_func=f_regression, k=min(8, X_train_full.shape[1]))
filter_selector.fit(X_fit_full, y_train_final)
filter_selected = X_train_full.columns[filter_selector.get_support()].tolist()
print(f"Filter selected: {filter_selected}")
#The filter method scores every feature independently by its relationship with 
#the target and keeps the 8 highest-scoring features.


In [ ]:
# wrapper: fit a random forest, remove the least important feature, repeat until 8 remain
estimator = RandomForestRegressor(n_estimators=100, random_state=42)
wrapper_selector = RFE(estimator, n_features_to_select=min(8, X_train_full.shape[1]))
wrapper_selector.fit(X_fit_full, y_train_final)
wrapper_selected = X_train_full.columns[wrapper_selector.support_].tolist()
print(f"Wrapper selected: {wrapper_selected}")


In [ ]:
selection_methods = {'Filter': filter_selected, 'Wrapper': wrapper_selected,
                     'All Features': X_train_full.columns.tolist()}
method_results = {}
for method_name, feats in selection_methods.items():
    selection_model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
    selection_model.fit(X_fit_full[feats], y_train_final)
    pred = selection_model.predict(X_val_full[feats])
    method_results[method_name] = {
        'rmse': root_mean_squared_error(y_val_final, pred),
        'mae': mean_absolute_error(y_val_final, pred),
        'r2': r2_score(y_val_final, pred), 'features': feats, 'n_features': len(feats)}
print('Feature-subset comparison on validation only:')
display(pd.DataFrame(method_results).T[['rmse','mae','r2','n_features']].astype(float).round(4))


In [ ]:
# Select the feature selection method with the lowest RMSE
best_method = min(method_results, key=lambda x: method_results[x]['rmse'])
selected_features = method_results[best_method]['features']

print(f"\nBEST METHOD: {best_method}")
print(f"   RMSE: {method_results[best_method]['rmse']:.2f}")
print(f"   Features ({len(selected_features)}): {selected_features}")

# Keep only the selected features for model training
X_train = X_train_full[selected_features]
X_test = X_test_full[selected_features]

#The feature set achieving the lowest Root Mean Squared Error (RMSE) is selected and applied to the training 
#and testing data used by the machine learning models.


In [ ]:
X_train_final = X_fit_full[selected_features]
X_val_final = X_val_full[selected_features]
X_test_final = X_test_full[selected_features]


#  Machine Learning Models

## CatBoost

In [ ]:
# candidate feature sets to compare
all_feats = X_train_full.columns.tolist()
calendar_trend = ["hour", "sin_hour", "cos_hour", "trend", "trend_squared"]
cycle_feats = [f"lag_{CYCLE}", f"ratio_lag1_lag{CYCLE}"]

candidate_sets = {
    "All features": all_feats,
    "All minus calendar/trend": [f for f in all_feats if f not in calendar_trend],
    "All minus cycle lag/ratio": [f for f in all_feats if f not in cycle_feats],
    "All minus both": [f for f in all_feats if f not in calendar_trend + cycle_feats],
    "Filter": filter_selected,
    "Wrapper": wrapper_selected,
}

# fit on the first 80% of training, score on validation; the test set is not used to choose
ablation = []
for name, feats in candidate_sets.items():
    model = CatBoostRegressor(iterations=5000, depth=6, learning_rate=0.03, loss_function="RMSE",
                              eval_metric="RMSE", random_seed=42, verbose=0)
    model.fit(X_train_full.iloc[:split][feats], y_train_final,
              eval_set=(X_train_full.iloc[split:][feats], y_val_final),
              early_stopping_rounds=100, use_best_model=True)
    val_rmse = root_mean_squared_error(y_val_final, model.predict(X_train_full.iloc[split:][feats]))
    ablation.append({"Feature set": name, "n_features": len(feats),
                     "Validation RMSE": val_rmse, "Best iteration": model.get_best_iteration()})

ablation = pd.DataFrame(ablation).sort_values("Validation RMSE").reset_index(drop=True)
display(ablation.round(4))

In [ ]:
best_set = candidate_sets[ablation.iloc[0]["Feature set"]]

X_train_final = X_train_full.iloc[:split][best_set]
X_val_final = X_train_full.iloc[split:][best_set]
X_test_final = X_test_full[best_set]

In [ ]:
# CatBoost model: up to 5000 trees, early stopping picks the best number
catboost = CatBoostRegressor(
    iterations=5000,
    depth=6,
    learning_rate=0.03,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42,
    verbose=500
)


# train on the first 80% of training, stop when validation RMSE stops improving for 100 rounds
catboost.fit(
    X_train_final, y_train_final,
    eval_set=(X_val_final, y_val_final),
    early_stopping_rounds=100,
    use_best_model=True
)
print("Best iteration:", catboost.get_best_iteration())

# validation predictions
y_val_pred_cb = catboost.predict(X_val_final)
print(f"Validation RMSE: {root_mean_squared_error(y_val_final, y_val_pred_cb):.4f}")

# test predictions
y_pred_cb = catboost.predict(X_test_final)
print(f"Test RMSE: {root_mean_squared_error(y_test_final, y_pred_cb):.4f}")

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

def regression_metrics(actual, predicted):
    # MAE: average error | MSE: average squared error | RMSE: MSE in the target's units | MAPE: error as % of the true value
    actual, predicted = np.asarray(actual, dtype=float), np.asarray(predicted, dtype=float)
    mse = np.mean((actual - predicted) ** 2)
    return {"MAE": np.mean(np.abs(actual - predicted)),
            "MSE": mse,
            "RMSE": np.sqrt(mse),
            "MAPE (%)": np.mean(np.abs((actual - predicted) / actual)) * 100}


def evaluate_ml_model(name, model, train_curve, val_curve, best_iter):
    # same tests and plots for every ML model, so results are directly comparable
    pred_train = model.predict(X_train_final)
    pred_val = model.predict(X_val_final)
    pred_test = model.predict(X_test_final)

    # 1. metrics on every split: similar validation and test errors indicate the model generalises
    table = pd.DataFrame({f"{name} (train)": regression_metrics(y_train_final, pred_train),
                          f"{name} (validation)": regression_metrics(y_val_final, pred_val),
                          f"{name} (test)": regression_metrics(y_test_final, pred_test)}).T
    display(table.round(4))

    # 2. learning curve: training stops at the tree with the lowest validation error
    plt.figure(figsize=(10, 5))
    plt.plot(train_curve, label="Training RMSE")
    plt.plot(val_curve, label="Validation RMSE")
    plt.axvline(best_iter, color="red", linestyle="--", label=f"Best iteration ({best_iter})")
    plt.title(f"{name}: learning curve")
    plt.xlabel("Number of trees"); plt.ylabel("RMSE (mm)")
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

    # 3. actual vs predicted: full test period and the first 4 hours
    fig, axes = plt.subplots(2, 1, figsize=(15, 8))
    axes[0].plot(y_test_final.index, y_test_final, label="Actual", linewidth=1)
    axes[0].plot(y_test_final.index, pred_test, label="Predicted", linewidth=1, alpha=0.8)
    axes[0].set_title(f"{name}: actual vs predicted (test period)")
    axes[1].plot(y_test_final.index[:240], y_test_final.iloc[:240], label="Actual", linewidth=1.5)
    axes[1].plot(y_test_final.index[:240], pred_test[:240], "--", label="Predicted", linewidth=1.5)
    axes[1].set_title(f"{name}: first 4 hours of the test period")
    for ax in axes:
        ax.set_ylabel("LIT101 (mm)"); ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

    # 4. residuals: should centre on zero with no pattern; ACF spikes mean structure the model missed
    residuals = y_test_final.values - pred_test
    fig, axes = plt.subplots(1, 2, figsize=(16, 4))
    axes[0].plot(y_test_final.index, residuals, linewidth=0.7)
    axes[0].axhline(0, color="red", linestyle="--")
    axes[0].set_title(f"{name}: test residuals over time (mean = {residuals.mean():.3f})")
    axes[0].set_ylabel("Actual - predicted (mm)")
    plot_acf(residuals, lags=2 * CYCLE + 5, ax=axes[1])
    axes[1].set_title(f"{name}: residual ACF")
    for ax in axes:
        ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

    # 5. feature importance
    pd.Series(model.feature_importances_, index=X_train_final.columns).sort_values().plot(kind="barh", figsize=(9, 5))
    plt.title(f"{name}: feature importance"); plt.xlabel("Importance")
    plt.tight_layout(); plt.show()

    return pred_test, table


# CatBoost evaluation
cb_history = catboost.get_evals_result()
y_pred_cb, cb_table = evaluate_ml_model(
    "CatBoost", catboost,
    train_curve=cb_history["learn"]["RMSE"],
    val_curve=cb_history["validation"]["RMSE"],
    best_iter=catboost.get_best_iteration(),
)

In [ ]:
# validation RMSE of every candidate feature set, best first
plt.figure(figsize=(10, 5))
plt.barh(ablation["Feature set"], ablation["Validation RMSE"], color="tab:blue")
plt.gca().invert_yaxis()
for i, v in enumerate(ablation["Validation RMSE"]):
    plt.text(v, i, f" {v:.2f}", va="center")
plt.title("CatBoost validation RMSE by feature set")
plt.xlabel("Validation RMSE")
plt.grid(True, axis="x", alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error
import pandas as pd
import numpy as np

cat_grid = {
    "max_depth": [3, 4, 6, 8],
    "learning_rate": [0.03, 0.06, 0.1],
    "subsample": [0.8, 1.0],
}

cat_rows = []

for depth in cat_grid["max_depth"]:
    for lr in cat_grid["learning_rate"]:
        for sub in cat_grid["subsample"]:

            model = CatBoostRegressor(
                iterations=5000,
                depth=depth,
                learning_rate=lr,
                subsample=sub,
                bootstrap_type="Bernoulli",
                loss_function="RMSE",
                eval_metric="RMSE",
                random_seed=42,
                verbose=False
            )

            model.fit(
                X_train_final,
                y_train_final,
                eval_set=(X_val_final, y_val_final),
                early_stopping_rounds=100,
                verbose=False
            )

            train_pred = model.predict(X_train_final)
            val_pred = model.predict(X_val_final)

            cat_rows.append({
                "max_depth": depth,
                "learning_rate": lr,
                "subsample": sub,
                "best_iteration": model.get_best_iteration(),
                "train_RMSE": np.sqrt(mean_squared_error(y_train_final, train_pred)),
                "val_RMSE": np.sqrt(mean_squared_error(y_val_final, val_pred))
            })

cat_tuning = (
    pd.DataFrame(cat_rows)
    .sort_values("val_RMSE")
    .reset_index(drop=True)
)

display(cat_tuning.head(10).round(4))

cat_best_params = {
    "depth": int(cat_tuning.loc[0, "max_depth"]),
    "learning_rate": float(cat_tuning.loc[0, "learning_rate"]),
    "subsample": float(cat_tuning.loc[0, "subsample"]),
    "iterations": int(cat_tuning.loc[0, "best_iteration"]) + 1,
    "bootstrap_type": "Bernoulli"
}

print("Best parameters:", cat_best_params)

In [ ]:
# best validation RMSE reached at each value of each hyperparameter
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, param in zip(axes, cat_grid):
    best_per_value = cat_tuning.groupby(param)["val_RMSE"].min()
    ax.plot(best_per_value.index.astype(str), best_per_value.values, "o-")
    ax.set_title(f"Validation RMSE vs {param}"); ax.set_xlabel(param); ax.grid(True, alpha=0.3)
axes[0].set_ylabel("Best validation RMSE (mm)")
plt.tight_layout(); plt.show()

# depth vs learning rate at the best subsample value
subset = cat_tuning[cat_tuning["subsample"] == cat_best_params["subsample"]]
plt.figure(figsize=(7, 5))
sns.heatmap(subset.pivot(index="max_depth", columns="learning_rate", values="val_RMSE"),
            annot=True, fmt=".2f", cmap="viridis_r")
plt.title(f"Validation RMSE: max_depth vs learning_rate (subsample = {cat_best_params['subsample']})")
plt.tight_layout(); plt.show()

In [ ]:
# Tuned CatBoost, tracking training and validation RMSE
cat_final = CatBoostRegressor(
    **{k: v for k, v in cat_best_params.items() if k != "iterations"},
    iterations=5000,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42
)

cat_final.fit(
    X_train_final,
    y_train_final,
    eval_set=[
        (X_train_final, y_train_final),
        (X_val_final, y_val_final)
    ],
    early_stopping_rounds=100,
    verbose=False
)

# CatBoost evaluation
cat_history = cat_final.get_evals_result()

y_pred_cat, cat_table = evaluate_ml_model(
    "CatBoost",
    cat_final,
    train_curve=cat_history["learn"]["RMSE"],
    val_curve=cat_history["validation_1"]["RMSE"],
    best_iter=cat_final.get_best_iteration()
)

In [ ]:
# Compare CatBoost models against the persistence baseline
# define y_persist
y_persist = X_test_full["lag_1"]

catboost_comparison = pd.DataFrame({
    "CatBoost (tuned)": regression_metrics(y_test_final, y_pred_cat),
    "CatBoost (default)": regression_metrics(y_test_final, y_pred_cb),
    "Persistence baseline": regression_metrics(y_test_final, y_persist),
}).T.sort_values("RMSE")

display(catboost_comparison.round(4))

# Visualize MAE and RMSE
catboost_comparison[["MAE", "RMSE"]].plot.barh(figsize=(9, 4))

plt.title("Test Error: CatBoost vs Persistence Baseline")
plt.xlabel("Error (mm)")
plt.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## XGBoost

In [ ]:
from xgboost import XGBRegressor

# XGBoost with default settings on the same features and split as CatBoost
xgb_default = XGBRegressor(n_estimators=100, random_state=42)
xgb_default.fit(X_train_final, y_train_final)
y_pred_xgb_default = xgb_default.predict(X_test_final)

display(pd.DataFrame({
    "XGBoost default (validation)": regression_metrics(y_val_final, xgb_default.predict(X_val_final)),
    "XGBoost default (test)": regression_metrics(y_test_final, y_pred_xgb_default),
}).T.round(4))

In [ ]:
# hyperparameters and the values tried for each
xgb_grid = {
    "max_depth": [3, 4, 6, 8],          # how deep each tree can grow
    "learning_rate": [0.03, 0.06, 0.1], # how strongly each tree corrects the previous ones
    "subsample": [0.8, 1.0],            # fraction of rows each tree sees (below 1 adds randomness)
}

xgb_rows = []
for depth in xgb_grid["max_depth"]:
    for lr in xgb_grid["learning_rate"]:
        for sub in xgb_grid["subsample"]:
            # up to 5000 trees; training stops when validation RMSE hasn't improved for 100 rounds
            model = XGBRegressor(n_estimators=5000, max_depth=depth, learning_rate=lr, subsample=sub,
                                 eval_metric="rmse", early_stopping_rounds=100, random_state=42)
            model.fit(X_train_final, y_train_final, eval_set=[(X_val_final, y_val_final)], verbose=False)
            xgb_rows.append({
                "max_depth": depth, "learning_rate": lr, "subsample": sub,
                "best_iteration": model.best_iteration,
                "train_RMSE": root_mean_squared_error(y_train_final, model.predict(X_train_final)),
                "val_RMSE": root_mean_squared_error(y_val_final, model.predict(X_val_final)),
            })

xgb_tuning = pd.DataFrame(xgb_rows).sort_values("val_RMSE").reset_index(drop=True)
display(xgb_tuning.head(10).round(4))

xgb_best_params = xgb_tuning.iloc[0][list(xgb_grid)].to_dict()
xgb_best_params["max_depth"] = int(xgb_best_params["max_depth"])
print("Best parameters:", xgb_best_params)

In [ ]:
# best validation RMSE reached at each value of each hyperparameter
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, param in zip(axes, xgb_grid):
    best_per_value = xgb_tuning.groupby(param)["val_RMSE"].min()
    ax.plot(best_per_value.index.astype(str), best_per_value.values, "o-")
    ax.set_title(f"Validation RMSE vs {param}"); ax.set_xlabel(param); ax.grid(True, alpha=0.3)
axes[0].set_ylabel("Best validation RMSE (mm)")
plt.tight_layout(); plt.show()

# depth vs learning rate at the best subsample value
subset = xgb_tuning[xgb_tuning["subsample"] == xgb_best_params["subsample"]]
plt.figure(figsize=(7, 5))
sns.heatmap(subset.pivot(index="max_depth", columns="learning_rate", values="val_RMSE"),
            annot=True, fmt=".2f", cmap="viridis_r")
plt.title(f"Validation RMSE: max_depth vs learning_rate (subsample = {xgb_best_params['subsample']})")
plt.tight_layout(); plt.show()

In [ ]:
# tuned XGBoost, tracking error on both sets after every tree
xgb_final = XGBRegressor(n_estimators=5000, **xgb_best_params, eval_metric="rmse",
                         early_stopping_rounds=100, random_state=42)
xgb_final.fit(X_train_final, y_train_final,
              eval_set=[(X_train_final, y_train_final), (X_val_final, y_val_final)], verbose=False)

# XGBoost evaluation
xgb_history = xgb_final.evals_result()
y_pred_xgb, xgb_table = evaluate_ml_model(
    "XGBoost", xgb_final,
    train_curve=xgb_history["validation_0"]["rmse"],
    val_curve=xgb_history["validation_1"]["rmse"],
    best_iter=xgb_final.best_iteration,
)

In [ ]:
# compare all ML models and the persistence baseline on the same test period
ml_comparison = pd.DataFrame({
    "CatBoost (tuned)": regression_metrics(y_test_final, y_pred_cat),
    "CatBoost (default)": regression_metrics(y_test_final, y_pred_cb),
    "XGBoost (tuned)": regression_metrics(y_test_final, y_pred_xgb),
    "XGBoost (default)": regression_metrics(y_test_final, y_pred_xgb_default),
    "Persistence baseline": regression_metrics(y_test_final, y_persist),
}).T.sort_values("RMSE")
display(ml_comparison.round(4))

ml_comparison[["MAE", "RMSE"]].plot.barh(figsize=(9, 4))
plt.title("Test error by ML model"); plt.xlabel("Error (mm)")
plt.grid(True, axis="x", alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# forecast the next 60 minutes after the last observation; each prediction is fed back in
# as history so the next minute's lags and rolling statistics can be built from it
def recursive_forecast(model, history, steps, feature_names):
    series = history[[target]].copy()
    for _ in range(steps):
        next_time = series.index[-1] + pd.Timedelta(minutes=1)
        series.loc[next_time, target] = np.nan
        next_row = create_all_features(series, drop_incomplete=False).iloc[[-1]][feature_names]
        series.loc[next_time, target] = float(model.predict(next_row)[0])
    return series[target].iloc[-steps:]

ml_features = list(X_train_final.columns)
future_xgb = recursive_forecast(xgb_final, ml_df, 60, ml_features)
future_cat = recursive_forecast(cat_final, ml_df, 60, ml_features)

plt.figure(figsize=(14, 5))
ml_df[target].iloc[-12 * 60:].plot(label="Recent normal-operation data")
future_xgb.plot(label="XGBoost next 60 minutes", linestyle="--")
future_cat.plot(label="CatBoost next 60 minutes", linestyle="--")
plt.title("LIT101: next 60 minutes, ML forecasts"); plt.ylabel("Water level (mm)")
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

# Statistical Models

## Univariate

In [ ]:

processed_data = processed_clean.copy() # LIT101

In [ ]:

#Univariate Statistical Forecasting ON ARIMA

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

from scipy.fft import fft, fftfreq
from sklearn.preprocessing import StandardScaler

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import acorr_ljungbox

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (12, 4)

uni_target = "LIT101"
uni_plot_hours = 12

uni_labels = (
    processed_clean["Normal/Attack"]
    .astype("string")
    .str.strip()
)

uni_attack_mask = uni_labels.eq("Attack")

if not uni_attack_mask.any():
    raise ValueError("No attack boundary found in processed_clean.")

uni_first_attack = processed_clean.index[uni_attack_mask].min()

uni_normal_data = processed_clean.loc[
    processed_clean.index < uni_first_attack
].copy()

uni_data = pd.to_numeric(
    uni_normal_data[uni_target],
    errors="coerce"
).sort_index()

if not uni_data.index.is_unique:
    raise ValueError(
        "Duplicate timestamps remain. Resolve them before ARIMA fitting."
    )

uni_data = uni_data.asfreq("1min")

print("Selected target:", uni_target)
print("Missing values:", uni_data.isna().sum())
print("Number of observations:", len(uni_data))
print("Start time:", uni_data.index.min())
print("End time:", uni_data.index.max())

display(uni_data.describe())

if uni_data.isna().any():
    raise ValueError(
        "Missing LIT101 readings detected. Investigate the gaps "
        "before fitting ARIMA; this section does not invent readings."
    )

uni_train_full_size = int(len(uni_data) * 0.8)

uni_train_full = uni_data.iloc[:uni_train_full_size].copy()
uni_test = uni_data.iloc[uni_train_full_size:].copy()

uni_inner_split = int(len(uni_train_full) * 0.8)

uni_train = uni_train_full.iloc[:uni_inner_split].copy()
uni_validation = uni_train_full.iloc[uni_inner_split:].copy()

print("\nTraining observations:", len(uni_train))
print("Validation observations:", len(uni_validation))
print("Full development observations:", len(uni_train_full))
print("Test observations:", len(uni_test))

print("\nTraining:", uni_train.index[0], "to", uni_train.index[-1])
print("Validation:", uni_validation.index[0], "to", uni_validation.index[-1])
print("Test:", uni_test.index[0], "to", uni_test.index[-1])


uni_fft_values = fft(
    (uni_train - uni_train.mean()).to_numpy()
)

uni_frequencies = fftfreq(
    len(uni_train),
    d=1
)

uni_band = (
    (uni_frequencies > 1 / 360)
    & (uni_frequencies < 1 / 5)
)

if not uni_band.any():
    raise ValueError("No suitable FFT frequencies were found.")

uni_period = int(round(
    1 / uni_frequencies[uni_band][
        np.argmax(np.abs(uni_fft_values[uni_band]))
    ]
))

print("\nDominant training cycle:", uni_period, "minutes")


In [ ]:
# Exploration
uni_plot_start = uni_train.index[-1] - pd.Timedelta(hours=uni_plot_hours)
uni_plot_end = uni_train.index[-1]

uni_train_zoom = uni_train.loc[uni_plot_start:uni_plot_end]

uni_rolling_mean = uni_train.rolling(window=69).mean()
uni_rolling_var = uni_train.rolling(window=69).var()

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

axes[0].plot(uni_train_zoom.index, uni_train_zoom, linewidth=0.8)
axes[0].set_title(f"{uni_target} - Observed Series (Last 69 Minutes)", loc="left")
axes[0].set_ylabel("Water Level")
axes[0].grid(True, linestyle="--", alpha=0.3)

axes[1].plot(
    uni_rolling_mean.loc[uni_plot_start:uni_plot_end].index,
    uni_rolling_mean.loc[uni_plot_start:uni_plot_end],
    linewidth=0.9
)
axes[1].set_title("12 hours Rolling Mean", loc="left")
axes[1].set_ylabel("Rolling Mean")
axes[1].grid(True, linestyle="--", alpha=0.3)

axes[2].plot(
    uni_rolling_var.loc[uni_plot_start:uni_plot_end].index,
    uni_rolling_var.loc[uni_plot_start:uni_plot_end],
    linewidth=0.9
)
axes[2].set_title("12 hours Rolling Variance", loc="left")
axes[2].set_ylabel("Rolling Variance")
axes[2].set_xlabel("Timestamp")
axes[2].grid(True, linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

uni_zscore = (uni_train - uni_train.mean()) / uni_train.std()

print(
    "Training observations with |z-score| > 3:",
    int((np.abs(uni_zscore) > 3).sum())
)

In [ ]:
#Decomposition

if uni_period >= 2 and 2 * uni_period < len(uni_train):

    uni_decomposition = seasonal_decompose(
        uni_train,
        model="additive",
        period=uni_period,
        extrapolate_trend="freq"
    )

    uni_components = pd.DataFrame({
        "Observed": uni_decomposition.observed,
        "Trend": uni_decomposition.trend,
        "Seasonal": uni_decomposition.seasonal,
        "Residual": uni_decomposition.resid
    })

    uni_zoom_start = (
        uni_train.index[-1]
        - pd.Timedelta(hours=uni_plot_hours)
    )

    uni_zoom = uni_components.loc[
        uni_zoom_start:uni_train.index[-1]
    ]

    uni_plot_settings = [
        ("Observed", "Observed LIT101", "Water Level"),
        ("Trend", "Trend Component", "Trend"),
        ("Seasonal", "Seasonal Component", "Seasonal"),
        ("Residual", "Residual Component", "Residual")
    ]

    fig, axes = plt.subplots(
        4, 1,
        figsize=(14, 9),
        sharex=True
    )

    for ax, (column, title, ylabel) in zip(
        axes, uni_plot_settings
    ):
        ax.plot(
            uni_zoom.index,
            uni_zoom[column],
            linewidth=0.8
        )
        ax.set_title(title, fontsize=11, loc="left")
        ax.set_ylabel(ylabel)
        ax.grid(True, linestyle="--", alpha=0.3)

    axes[-1].set_xlabel("Timestamp")

    fig.suptitle(
        f"{uni_target} - Time-Series Decomposition (12 Hours)",
        fontsize=14
    )

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

    print("Decomposition period:", uni_period, "minutes")

else:
    print("The detected cycle is unsuitable for decomposition.")


In [ ]:
#Stationarity Check

def uni_check_stationarity(series, title):

    series = series.dropna()

    print(f"\n{title}")
    print("-" * 50)

    uni_start = (
        series.index[-1]
        - pd.Timedelta(hours=uni_plot_hours)
    )

    uni_zoom_series = series.loc[
        uni_start:series.index[-1]
    ]

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(
        uni_zoom_series.index,
        uni_zoom_series,
        linewidth=0.8
    )
    ax.set_title(f"{title} - Last 12 Hours", loc="left")
    ax.set_xlabel("Timestamp")
    ax.set_ylabel("Value")
    ax.grid(True, linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(12, 4))
    plot_acf(series, lags=30, ax=ax)
    ax.set_title(f"{title} - ACF (30 Lags)")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    uni_adf = adfuller(series, autolag="AIC")

    print(f"ADF Statistic: {uni_adf[0]:.4f}")
    print(f"ADF p-value: {uni_adf[1]:.4f}")
    print(f"ADF supports stationarity: {uni_adf[1] <= 0.05}")

    uni_kpss = kpss(
        series,
        regression="c",
        nlags="auto"
    )

    print(f"KPSS Statistic: {uni_kpss[0]:.4f}")
    print(f"KPSS p-value: {uni_kpss[1]:.4f}")
    print(f"KPSS supports stationarity: {uni_kpss[1] > 0.05}")

    return uni_adf, uni_kpss


uni_adf_original, uni_kpss_original = uni_check_stationarity(
    uni_train,
    f"{uni_target} - Original Training Series"
)


In [ ]:
#Transformation

uni_diff = uni_train.diff().dropna()

uni_adf_diff, uni_kpss_diff = uni_check_stationarity(
    uni_diff,
    f"{uni_target} - First-Differenced Series"
)

uni_original_stationary = (
    uni_adf_original[1] <= 0.05
    and uni_kpss_original[1] > 0.05
)

uni_diff_stationary = (
    uni_adf_diff[1] <= 0.05
    and uni_kpss_diff[1] > 0.05
)

if uni_original_stationary:
    uni_d = 0

elif uni_diff_stationary:
    uni_d = 1

else:
    uni_d = 1

    print(
        "\nWarning: Stationarity tests are inconclusive. "
        "Using d=1 as a candidate; interpret results cautiously."
    )

print("\nSelected ARIMA differencing order d =", uni_d)


In [ ]:


#Normalization

uni_scaler = StandardScaler()

uni_train_scaled = pd.Series(
    uni_scaler.fit_transform(
        uni_train.to_numpy().reshape(-1, 1)
    ).ravel(),
    index=uni_train.index,
    name=uni_target
)

print("Original training mean:", round(uni_train.mean(), 4))
print("Original training standard deviation:", round(uni_train.std(), 4))
print("Scaled training mean:", round(uni_train_scaled.mean(), 4))
print("Scaled training standard deviation:", round(uni_train_scaled.std(), 4))

#Feature Understanding

if uni_d == 0:
    uni_structure_series = uni_train_scaled
else:
    uni_structure_series = uni_train_scaled.diff().dropna()

fig, axes = plt.subplots(
    2, 1,
    figsize=(12, 7)
)

plot_acf(
    uni_structure_series,
    lags=40,
    ax=axes[0]
)
axes[0].set_title("ACF - Stationarity-Adjusted LIT101")
axes[0].grid(True, alpha=0.3)

plot_pacf(
    uni_structure_series,
    lags=40,
    ax=axes[1]
)
axes[1].set_title("PACF - Stationarity-Adjusted LIT101")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("ACF helps suggest the moving-average order q.")
print("PACF helps suggest the autoregressive order p.")
print("Stationarity tests and differencing guide the order d.")


In [ ]:
# Model Selection & Training
# One-step-ahead forecasting 
uni_trend = "c" if uni_d == 0 else "n"

uni_candidate_orders = [
    (4, 0, 2),
    (8, 0, 2),
    (10, 0, 1),
    (10, 0, 2),
    (12, 0, 1),
    (12, 0, 2),
    (14, 0, 1),
    (14, 0, 2)
]

uni_candidate_fits = {}
uni_model_scores = []


def uni_calculate_metrics(actual, predicted):

    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    uni_mae_value = np.mean(np.abs(actual - predicted))
    uni_mse_value = np.mean((actual - predicted) ** 2)
    uni_rmse_value = np.sqrt(uni_mse_value)

    uni_nonzero = np.abs(actual) > 1e-8

    if uni_nonzero.any():
        uni_mape_value = np.mean(
            np.abs(
                (actual[uni_nonzero] - predicted[uni_nonzero])
                / actual[uni_nonzero]
            )
        ) * 100
    else:
        uni_mape_value = np.nan

    return {
        "MAE": uni_mae_value,
        "MSE": uni_mse_value,
        "RMSE": uni_rmse_value,
        "MAPE (%)": uni_mape_value
    }


# Scale validation once
uni_val_scaled = uni_scaler.transform(
    uni_validation.to_numpy().reshape(-1, 1)
).ravel()

uni_val_actual = uni_validation.to_numpy()


for uni_order in uni_candidate_orders:

    uni_name = f"ARIMA{uni_order}"

    print(f"\nFitting {uni_name}...")

    try:
        uni_fit = ARIMA(
            uni_train_scaled,
            order=uni_order,
            trend=uni_trend
        ).fit()

        # One-step-ahead rolling validation
        uni_predictions_scaled = []
        uni_updated_fit = uni_fit

        for actual_value in uni_val_scaled:

            # Predict next minute
            uni_prediction = np.asarray(
                uni_updated_fit.forecast(steps=1)
            ).ravel()[0]

            uni_predictions_scaled.append(uni_prediction)

            # Incorporate actual observation without refitting
            uni_updated_fit = uni_updated_fit.append(
                [actual_value],
                refit=False
            )

        # Inverse transform predictions
        uni_val_forecast = uni_scaler.inverse_transform(
            np.asarray(uni_predictions_scaled).reshape(-1, 1)
        ).ravel()

        # Calculate validation metrics
        uni_val_metrics = uni_calculate_metrics(
            uni_val_actual,
            uni_val_forecast
        )

        # Residual diagnostics
        uni_candidate_resid = np.asarray(uni_fit.resid)
        uni_model_df = uni_order[0] + uni_order[2]

        uni_candidate_lb = acorr_ljungbox(
            uni_candidate_resid,
            lags=[69, 138],
            model_df=uni_model_df,
            return_df=True
        )

        uni_lb_pvalue = float(
            uni_candidate_lb["lb_pvalue"].min()
        )

        # Save fitted model
        uni_candidate_fits[uni_order] = uni_fit

        uni_model_scores.append({
            "Model": uni_name,
            "Order": uni_order,
            "AIC": uni_fit.aic,
            "BIC": uni_fit.bic,
            "Validation MAE": uni_val_metrics["MAE"],
            "Validation MSE": uni_val_metrics["MSE"],
            "Validation RMSE": uni_val_metrics["RMSE"],
            "Validation MAPE (%)": uni_val_metrics["MAPE (%)"],
            "Ljung-Box p-value": uni_lb_pvalue,
            "Residuals pass test": uni_lb_pvalue > 0.05
        })

        print(f"AIC: {uni_fit.aic:.2f}")
        print(f"BIC: {uni_fit.bic:.2f}")
        print(f"Validation MAE: {uni_val_metrics['MAE']:.4f}")
        print(f"Validation MSE: {uni_val_metrics['MSE']:.4f}")
        print(f"Validation RMSE: {uni_val_metrics['RMSE']:.4f}")
        print(f"Validation MAPE: {uni_val_metrics['MAPE (%)']:.2f}%")
        print(f"Ljung-Box p-value: {uni_lb_pvalue:.6g}")

    except Exception as error:
        print(f"{uni_name} failed: {error}")


if not uni_model_scores:
    raise RuntimeError(
        "No ARIMA candidate fitted successfully. "
        "Check the data and stationarity decision."
    )


# Model comparison
uni_comparison = pd.DataFrame(uni_model_scores)

display(
    uni_comparison
    .sort_values("Validation RMSE")
    .round(4)
)


# Select best model
uni_white_noise_candidates = [
    row for row in uni_model_scores
    if row["Residuals pass test"]
]

if uni_white_noise_candidates:

    uni_best_row = min(
        uni_white_noise_candidates,
        key=lambda row: row["Validation RMSE"]
    )

    print(
        "\nSelected from candidates passing the residual test, "
        "using lowest one-step validation RMSE."
    )

else:

    uni_best_row = min(
        uni_model_scores,
        key=lambda row: row["Validation RMSE"]
    )

    print(
        "\nWarning: None of the tested ARIMA candidates passed "
        "the residual white-noise test. The lowest-validation-RMSE "
        "candidate is retained, but model limitations must be reported."
    )


uni_best_order = uni_best_row["Order"]
uni_best_name = uni_best_row["Model"]

print("\nSelected model:", uni_best_name)
print("Selected order:", uni_best_order)
print("One-step Validation RMSE:", round(uni_best_row["Validation RMSE"], 4))

In [ ]:
# Compare AIC and BIC
uni_best_aic = min(uni_model_scores, key=lambda x: x["AIC"])
uni_best_bic = min(uni_model_scores, key=lambda x: x["BIC"])

print("Best by AIC:", uni_best_aic["Model"])
print("Best by BIC:", uni_best_bic["Model"])

# Select based on BIC
uni_best_row = uni_best_bic

uni_best_order = uni_best_row["Order"]
uni_best_name = uni_best_row["Model"]

# Retrieve already fitted model
uni_best_fit = uni_candidate_fits[uni_best_order]

print("\nFINAL SELECTED MODEL")
print("Model:", uni_best_name)
print("Order:", uni_best_order)
print("AIC:", round(uni_best_row["AIC"], 2))
print("BIC:", round(uni_best_row["BIC"], 2))
print("Validation RMSE:", round(uni_best_row["Validation RMSE"], 4))
print("Ljung-Box p-value:", uni_best_row["Ljung-Box p-value"])

# Display comparison
display(
    uni_comparison.sort_values("BIC").round(4)
)

In [ ]:

# Moving Average (MA), Weighted Moving Average (WMA), Exponential Moving Average (EMA)
# One-step-ahead forecasts using only previous observations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Use the same univariate LIT101 series and chronological splits
uni_smoothing_data = uni_data.astype(float)
uni_smoothing_train_val = pd.concat([uni_train, uni_validation])
def smoothing_predictions(series, method, window=None, alpha=None):
    previous = series.shift(1)

    if method == "MA":
        return previous.rolling(window=window).mean()

    elif method == "WMA":
        weights = np.arange(1, window + 1, dtype=float)
        return previous.rolling(window=window).apply(
            lambda x: np.dot(x, weights) / weights.sum(),
            raw=True
        )

    elif method == "EMA":
        return previous.ewm(alpha=alpha, adjust=False).mean()

# Tune each model on validation RMSE
smoothing_results = []

for method in ["MA", "WMA", "EMA"]:

    parameters = (
        [3, 6, 12, CYCLE]
        if method in ["MA", "WMA"]
        else [0.1, 0.3, 0.5, 0.8]
    )

    for param in parameters:
        pred = smoothing_predictions(
            uni_smoothing_train_val,
            method,
            window=param if method != "EMA" else None,
            alpha=param if method == "EMA" else None
        ).loc[uni_validation.index]

        metrics = regression_metrics(uni_validation, pred)

        smoothing_results.append({
            "Model": method,
            "Parameter": param,
            "Validation RMSE": metrics["RMSE"]
        })

smoothing_results = pd.DataFrame(smoothing_results)

# Best parameter for each smoothing model
smoothing_best = (
    smoothing_results
    .sort_values("Validation RMSE")
    .drop_duplicates("Model")
    .reset_index(drop=True)
)

print("Best validation settings:")
display(smoothing_best.round(4))

# Evaluate each selected model on the test set
smoothing_test_predictions = {}
smoothing_test_results = []

for _, row in smoothing_best.iterrows():
    method = row["Model"]
    param = row["Parameter"]

    pred = smoothing_predictions(
        uni_smoothing_data,
        method,
        window=int(param) if method != "EMA" else None,
        alpha=float(param) if method == "EMA" else None
    ).loc[uni_test.index]

    smoothing_test_predictions[method] = pred

    metrics = regression_metrics(uni_test, pred)
    smoothing_test_results.append({
        "Model": method,
        "Parameter": param,
        **metrics
    })

smoothing_test_results = (
    pd.DataFrame(smoothing_test_results)
    .sort_values("RMSE")
    .reset_index(drop=True)
)

print("\nTest set comparison:")
display(smoothing_test_results.round(4))

# Best smoothing baseline
uni_smooth_name = smoothing_test_results.iloc[0]["Model"]
uni_smooth_pred = smoothing_test_predictions[uni_smooth_name]

print(f"\nBest smoothing baseline: {uni_smooth_name}")

# Plot test predictions (first 12 hours)
plot_index = uni_test.index[:12 * 60]

plt.figure(figsize=(14, 5))
plt.plot(plot_index, uni_test.loc[plot_index],
         label="Actual LIT101", color="black", linewidth=1.5)

for name, pred in smoothing_test_predictions.items():
    plt.plot(plot_index, pred.loc[plot_index],
             label=name, alpha=0.8)

plt.title("MA vs WMA vs EMA: One-Step Forecasting")
plt.xlabel("Timestamp")
plt.ylabel("LIT101")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Compare test RMSE
plt.figure(figsize=(8, 4))
plt.bar(smoothing_test_results["Model"], smoothing_test_results["RMSE"])
plt.title("Smoothing Models: Test RMSE")
plt.ylabel("RMSE")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


ema is the best

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 5))
plot_acf(uni_train_scaled, lags=240)
plt.title("LIT101 ACF: Checking for Seasonality")
plt.show()

In [ ]:

series = np.asarray(uni_train_scaled).ravel()
series = series - np.mean(series)

fft_values = np.fft.rfft(series)
frequencies = np.fft.rfftfreq(len(series), d=1)

power = np.abs(fft_values) ** 2

# Exclude zero frequency
valid = frequencies > 0

periods = 1 / frequencies[valid]
powers = power[valid]

# Investigate periods from 4 to 240 minutes
mask = (periods >= 4) & (periods <= 240)

top = pd.DataFrame({
    "Period (minutes)": periods[mask],
    "Power": powers[mask]
}).nlargest(10, "Power")

display(top)

plt.figure(figsize=(12, 4))
plt.plot(periods[mask], powers[mask])
plt.xlabel("Period (minutes)")
plt.ylabel("Spectral power")
plt.title("LIT101 FFT: Dominant Periods")
plt.grid(alpha=0.3)
plt.show()

print("Previously selected FFT period:", uni_period)

In [ ]:
#White Noise Check

uni_final_scaler = StandardScaler()

uni_train_full_scaled = pd.Series(
    uni_final_scaler.fit_transform(
        uni_train_full.to_numpy().reshape(-1, 1)
    ).ravel(),
    index=uni_train_full.index,
    name=uni_target
)

uni_best_model = ARIMA(
    uni_train_full_scaled,
    order=uni_best_order,
    trend=uni_trend
).fit()

uni_residuals = pd.Series(
    np.asarray(uni_best_model.resid)
    * uni_final_scaler.scale_[0],
    index=uni_train_full.index,
    name="Residual"
).dropna()

print("Selected model:", uni_best_name)
print("Training AIC:", round(uni_best_model.aic, 4))
print("Training BIC:", round(uni_best_model.bic, 4))
print("Residual mean:", round(uni_residuals.mean(), 4))
print("Residual variance:", round(uni_residuals.var(), 4))

uni_resid_start = (
    uni_residuals.index[-1]
    - pd.Timedelta(hours=uni_plot_hours)
)

uni_residual_zoom = uni_residuals.loc[
    uni_resid_start:uni_residuals.index[-1]
]

uni_rolling_resid_var = uni_residuals.rolling(
    window=60
).var()

uni_resid_var_zoom = uni_rolling_resid_var.loc[
    uni_resid_start:uni_residuals.index[-1]
]

fig, axes = plt.subplots(
    3, 1,
    figsize=(14, 9)
)

axes[0].plot(
    uni_residual_zoom.index,
    uni_residual_zoom,
    linewidth=0.8
)
axes[0].axhline(
    0,
    color="red",
    linestyle="--",
    alpha=0.5
)
axes[0].set_title("ARIMA Residuals - Last 12 Hours", loc="left")
axes[0].set_ylabel("Residual")
axes[0].grid(True, alpha=0.3)

axes[1].plot(
    uni_resid_var_zoom.index,
    uni_resid_var_zoom,
    linewidth=0.8
)
axes[1].set_title(
    "Residual Rolling Variance - Last 12 Hours",
    loc="left"
)
axes[1].set_ylabel("Variance")
axes[1].grid(True, alpha=0.3)

plot_acf(
    uni_residuals,
    lags=69,
    ax=axes[2]
)
axes[2].set_title("Residual ACF - 30 Lags")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

uni_model_df = uni_best_order[0] + uni_best_order[2]

uni_lb_result = acorr_ljungbox(
    uni_residuals,
    lags=[69, 2*69 ],
    model_df=uni_model_df,
    return_df=True
)

display(uni_lb_result)

if uni_lb_result["lb_pvalue"].iloc[0] > 0.05:
    print(
        "No significant residual autocorrelation detected "
        "at the tested lag."
    )
else:
    print(
        "Significant residual autocorrelation remains. "
        "The selected ARIMA model does not fully capture "
        "the time-series structure."
    )


In [ ]:
# Model Evaluation
uni_to_mm = lambda a: uni_final_scaler.inverse_transform(np.asarray(a).reshape(-1, 1)).ravel()

# multi-step: the whole test period forecast from the end of the training data
uni_test_result = uni_best_model.get_forecast(steps=len(uni_test))
uni_test_forecast = pd.Series(uni_to_mm(uni_test_result.predicted_mean), index=uni_test.index, name="ARIMA Forecast")
uni_test_ci = pd.DataFrame({"Lower 95% CI": uni_to_mm(uni_test_result.conf_int().iloc[:, 0]),
                            "Upper 95% CI": uni_to_mm(uni_test_result.conf_int().iloc[:, 1])}, index=uni_test.index)

# one-step: parameters fixed, each test minute predicted from the observations up to the previous minute
uni_test_scaled = pd.Series(uni_final_scaler.transform(uni_test.to_numpy().reshape(-1, 1)).ravel(),
                            index=uni_test.index, name=uni_target)
uni_onestep = pd.Series(uni_to_mm(uni_best_model.append(uni_test_scaled, refit=False).fittedvalues.loc[uni_test.index]),
                        index=uni_test.index)

display(pd.DataFrame({
    f"{uni_best_name} multi-step": regression_metrics(uni_test, uni_test_forecast),
    f"{uni_best_name} one-step": regression_metrics(uni_test, uni_onestep),
}).T.round(4))

In [ ]:
#Phase Forecast

uni_test_plot_end = (
    uni_test.index[0]
    + pd.Timedelta(hours=uni_plot_hours)
)

uni_test_zoom = uni_test.loc[
    uni_test.index <= uni_test_plot_end
]

uni_forecast_zoom = uni_test_forecast.loc[
    uni_test_zoom.index
]

uni_ci_zoom = uni_test_ci.loc[
    uni_test_zoom.index
]

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(
    uni_test_zoom.index,
    uni_test_zoom,
    label="Actual Test Data",
    linewidth=1
)

ax.plot(
    uni_forecast_zoom.index,
    uni_forecast_zoom,
    label=f"{uni_best_name} Forecast",
    linestyle="--",
    linewidth=1
)

ax.fill_between(
    uni_test_zoom.index,
    uni_ci_zoom["Lower 95% CI"].to_numpy(),
    uni_ci_zoom["Upper 95% CI"].to_numpy(),
    alpha=0.2,
    label="95% Confidence Interval"
)

ax.set_title(
    f"{uni_target} - ARIMA Forecast vs Actual (12 Hours)",
    loc="left"
)
ax.set_xlabel("Timestamp")
ax.set_ylabel("Water Level")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:

#Finalforecasting STEP

uni_full_scaler = StandardScaler()

uni_full_scaled = pd.Series(
    uni_full_scaler.fit_transform(
        uni_data.to_numpy().reshape(-1, 1)
    ).ravel(),
    index=uni_data.index,
    name=uni_target
)

uni_final_model = ARIMA(
    uni_full_scaled,
    order=uni_best_order,
    trend=uni_trend
).fit()

uni_future_result = uni_final_model.get_forecast(
    steps=60
)

uni_future_values = uni_full_scaler.inverse_transform(
    np.asarray(
        uni_future_result.predicted_mean
    ).reshape(-1, 1)
).ravel()

uni_future_index = pd.date_range(
    start=uni_data.index[-1] + pd.Timedelta(minutes=1),
    periods=60,
    freq="1min"
)

uni_future_forecast = pd.Series(
    uni_future_values,
    index=uni_future_index,
    name="Forecast"
)

uni_future_ci_scaled = uni_future_result.conf_int()

uni_future_lower = uni_full_scaler.inverse_transform(
    uni_future_ci_scaled.iloc[:, 0].to_numpy().reshape(-1, 1)
).ravel()

uni_future_upper = uni_full_scaler.inverse_transform(
    uni_future_ci_scaled.iloc[:, 1].to_numpy().reshape(-1, 1)
).ravel()

uni_future_ci = pd.DataFrame({
    "Lower 95% CI": uni_future_lower,
    "Upper 95% CI": uni_future_upper
}, index=uni_future_index)

uni_history_start = (
    uni_data.index[-1]
    - pd.Timedelta(hours=uni_plot_hours)
)

uni_history_zoom = uni_data.loc[
    uni_data.index >= uni_history_start
]

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(
    uni_history_zoom.index,
    uni_history_zoom,
    label="Recent Normal-Operation Data",
    linewidth=0.9
)

ax.plot(
    uni_future_forecast.index,
    uni_future_forecast,
    label="Next 60 Minutes Forecast",
    linestyle="--",
    linewidth=1.3
)

ax.fill_between(
    uni_future_index,
    uni_future_ci["Lower 95% CI"].to_numpy(),
    uni_future_ci["Upper 95% CI"].to_numpy(),
    alpha=0.2,
    label="95% Confidence Interval"
)

ax.set_title(
    f"{uni_target} - Next 60 Minutes ARIMA Forecast",
    loc="left"
)
ax.set_xlabel("Timestamp")
ax.set_ylabel("Water Level")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

display(
    pd.DataFrame({
        "Forecast": uni_future_forecast,
        "Lower 95% CI": uni_future_ci["Lower 95% CI"],
        "Upper 95% CI": uni_future_ci["Upper 95% CI"]
    }).head(10).round(4)
)


## Multivariate

In [ ]:
var_target = "LIT101"
var_fit_raw = normal_1min.loc[normal_1min.index < VAL_START]

screen_rows = []
for col in continuous_cols:
    if col == var_target or var_fit_raw[col].std() == 0:
        continue
    x = var_fit_raw[col]
    adf_p = adfuller(x, autolag="AIC")[1]
    kpss_p = kpss(x, regression="c", nlags="auto")[1]
    # correlation between LIT101 now and the sensor k minutes earlier (k = 1 ... one cycle),
    # because the VAR predicts LIT101 from PAST sensor values
    lagged_r = [var_fit_raw[var_target].corr(x.shift(k)) for k in range(1, CYCLE + 1)]
    best_k = int(np.nanargmax(np.abs(lagged_r))) + 1
    screen_rows.append({"Sensor": col, "Best lag (min)": best_k, "Lagged corr": lagged_r[best_k - 1],
                        "|Lagged corr|": abs(lagged_r[best_k - 1]), "ADF p": adf_p, "KPSS p": kpss_p,
                        "Stationary": (adf_p <= 0.05) and (kpss_p > 0.05)})
var_screen = pd.DataFrame(screen_rows).sort_values("|Lagged corr|", ascending=False).reset_index(drop=True)

# redundancy: a stationary sensor that is > 0.9 correlated with a higher-ranked one is dropped
REDUNDANCY_LIMIT = 0.9
var_screen["Redundant with"] = ""
var_candidates = []
for i, row in var_screen.iterrows():
    if not row["Stationary"]:
        continue
    twin = [k for k in var_candidates if abs(var_fit_raw[row["Sensor"]].corr(var_fit_raw[k])) > REDUNDANCY_LIMIT]
    if twin:
        var_screen.loc[i, "Redundant with"] = twin[0]
    else:
        var_candidates.append(row["Sensor"])

display(var_screen.round(4))

In [ ]:
var_fit_mask = normal_1min.index < VAL_START
var_val_index = normal_1min.index[(normal_1min.index >= VAL_START) & (normal_1min.index < TEST_START)]
var_test_index = normal_1min.index[normal_1min.index >= TEST_START]

def var_one_step(model, data, index):
    """One-step-ahead forecast of column 0 for each minute in `index`, using the true previous k_ar minutes.
    Same result as calling model.forecast minute by minute, computed as one matrix operation."""
    y = data.to_numpy()
    pos = data.index.get_indexer(index)
    pred = np.full(len(pos), model.intercept[0])
    for i in range(model.k_ar):
        pred += y[pos - i - 1] @ model.coefs[i][0]   # coefs[i][0] = target equation's weights at lag i+1
    return pd.Series(pred, index=index)

def zscore(df, fit_mask):
    """z-score using mean/std learned on the fit period only, so validation/test stay unseen."""
    mean, std = df[fit_mask].mean(), df[fit_mask].std()
    return (df - mean) / std, mean, std

def score_set(sensors, lag):
    cols = [var_target, *sensors]
    data, mean, std = zscore(normal_1min[cols], var_fit_mask)
    model = VAR(data[var_fit_mask]).fit(lag)
    val_pred = var_one_step(model, data, var_val_index) * std[var_target] + mean[var_target]   # back to mm
    m = regression_metrics(normal_1min.loc[var_val_index, var_target], val_pred)
    return {"N sensors": len(sensors), "Sensors": ", ".join(sensors), "Lag": lag,
            "AIC": model.aic, "BIC": model.bic, "Val MAE": m["MAE"],
            "Val RMSE": m["RMSE"], "Val MAPE (%)": m["MAPE (%)"]}

# lag order chosen once by AIC on the full candidate set (search up to one fill/drain cycle)
full_data, _, _ = zscore(normal_1min[[var_target, *var_candidates]], var_fit_mask)
var_lag = int(VAR(full_data[var_fit_mask]).select_order(maxlags=CYCLE).aic)
print("Lag chosen by AIC:", var_lag)

# forward selection: at each step add the sensor that lowers validation RMSE the most
selected, remaining, ablation_rows = [], list(var_candidates), []
while remaining:
    step_rows = [score_set(selected + [s], var_lag) for s in remaining]
    ablation_rows.extend(step_rows)
    best = min(step_rows, key=lambda r: r["Val RMSE"])
    added = best["Sensors"].split(", ")[-1]
    selected.append(added); remaining.remove(added)

var_ablation = pd.DataFrame(ablation_rows).sort_values(["N sensors", "Val RMSE"]).reset_index(drop=True)
display(var_ablation.round(4))

var_best_per_size = var_ablation.loc[var_ablation.groupby("N sensors")["Val RMSE"].idxmin()].reset_index(drop=True)
var_best_per_size["Change vs previous size"] = var_best_per_size["Val RMSE"].diff()
display(var_best_per_size.round(4))

best_row = var_best_per_size.loc[var_best_per_size["Val RMSE"].idxmin()]
var_columns = [var_target] + best_row["Sensors"].split(", ")
print(f"Selected (lowest validation RMSE): {var_columns}, lag {var_lag}")

plt.figure(figsize=(10, 5))
plt.plot(var_best_per_size["N sensors"], var_best_per_size["Val RMSE"], "o-")
for _, r in var_best_per_size.iterrows():
    plt.annotate(f"{r['Val RMSE']:.3f}", (r["N sensors"], r["Val RMSE"]), textcoords="offset points", xytext=(0, 8), ha="center")
plt.title("VAR forward selection: best validation RMSE for each number of sensors")
plt.xlabel("Number of sensors (plus the LIT101 target)"); plt.ylabel("Validation RMSE (mm)")
plt.xticks(var_best_per_size["N sensors"]); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
var_d = 0   # all selected series passed ADF and KPSS in step 1, so no differencing
var_train_mask = normal_1min.index < TEST_START
var_data, var_mean, var_std = zscore(normal_1min[var_columns], var_train_mask)
var_model = VAR(var_data[var_train_mask]).fit(var_lag)
print("Columns:", var_columns, "| lag:", var_lag, "| stable:", var_model.is_stable())

var_pred = var_one_step(var_model, var_data, var_test_index) * var_std[var_target] + var_mean[var_target]
var_actual = normal_1min.loc[var_test_index, var_target]
display(pd.DataFrame([regression_metrics(var_actual, var_pred)], index=["VAR (one-step)"]).round(4))

plt.figure(figsize=(14, 5))
plt.plot(var_test_index[:240], var_actual.iloc[:240], label="Actual")
plt.plot(var_test_index[:240], var_pred.iloc[:240], "--", label="VAR one-step forecast")
plt.title("VAR: actual vs predicted LIT101 (first 4 hours of test)")
plt.xlabel("Time"); plt.ylabel("Water level (mm)")
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
from statsmodels.stats.diagnostic import het_arch
from statsmodels.tsa.stattools import acf

def white_noise_summary(resid, lag):

    lb_corr = acorr_ljungbox(resid, lags=[lag + 20, lag + CYCLE], model_df=lag, return_df=True)["lb_pvalue"].values
    lb_plain = acorr_ljungbox(resid, lags=[2 * CYCLE], return_df=True)["lb_pvalue"].values[0]
    roll = resid.rolling(CYCLE).var()
    acf_vals = acf(resid, nlags=2 * CYCLE)[1:]
    return {"Mean": resid.mean(),
            "Rolling var max/min": roll.max() / roll.min(),
            "ARCH p (variance)": het_arch(resid, nlags=10)[1],
            "Max |ACF|": np.abs(acf_vals).max(),
            f"LB plain p (lag {2*CYCLE})": lb_plain,
            "LB corrected p (lag+20)": lb_corr[0],
            f"LB corrected p (lag+{CYCLE})": lb_corr[1]}

var_resid = var_model.resid[var_target] * var_std[var_target]   # back to mm
var_wn = white_noise_summary(var_resid, var_lag)

fig, axes = plt.subplots(3, 1, figsize=(14, 10))
axes[0].plot(var_resid.iloc[-uni_plot_hours * 60:], linewidth=0.7); axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_title("1) Residuals (last 12 hours of training): should centre on 0")
axes[1].plot(var_resid.rolling(CYCLE).var(), linewidth=0.7)
axes[1].set_title("2) 69-minute rolling variance: should be roughly flat")
plot_acf(var_resid, lags=2 * CYCLE + 5, ax=axes[2]); axes[2].set_title("3) Residual ACF: bars should stay inside the band")
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

display(pd.Series(var_wn).round(6))


In [ ]:
from arch import arch_model

wn_raw = normal_1min[var_columns].astype(float)
y_mm = normal_1min[var_target]

def deseasonalise(df):
    # seasonal profile of one CYCLE learned on the fit period only, then subtracted everywhere
    pos = np.arange(len(df)) % CYCLE
    prof = {c: seasonal_decompose(df.loc[var_fit_mask, c], period=CYCLE, model="additive",
                                  extrapolate_trend="freq").seasonal.to_numpy()[:CYCLE][pos] for c in df}
    prof = pd.DataFrame(prof, index=df.index)
    return df - prof, prof[var_target]

des_data, des_profile = deseasonalise(wn_raw)
# each attempt: (transformed data, function that turns target predictions back into mm)
attempts = {
    "Levels (no transform)":   (wn_raw,                         lambda s: s),
    "First difference":        (wn_raw.diff().dropna(),         lambda s: s + y_mm.shift(1).loc[s.index]),
    f"Seasonal diff ({CYCLE})": (wn_raw.diff(CYCLE).dropna(),   lambda s: s + y_mm.shift(CYCLE).loc[s.index]),
    f"Deseasonalised ({CYCLE})": (des_data,                     lambda s: s + des_profile.loc[s.index]),
    "Log (log1p)":             (np.log1p(wn_raw),               lambda s: np.expm1(s)),
    "Plus actuator P101": (
        normal_1min[var_columns + ["P101"]].astype(float), lambda s: s),
}
lag_grid = [20, 40, CYCLE, 100, 150]

wn_rows = []
for name, (data, back) in attempts.items():
    fit_part = data[data.index < VAL_START]
    for lag in lag_grid:
        model = VAR(fit_part).fit(lag)
        resid = model.resid[var_target]
        val_rmse = regression_metrics(y_mm.loc[var_val_index], back(var_one_step(model, data, var_val_index)))["RMSE"]
        wn_rows.append({"Attempt": name, "Lag": lag, "Val RMSE": val_rmse, **white_noise_summary(resid, lag)})
        if name == "Levels (no transform)":
            # GARCH(1,1) models the changing variance; we re-check the standardised residuals
            z = arch_model(resid, mean="Zero", vol="GARCH", p=1, q=1, rescale=False).fit(disp="off").std_resid.dropna()
            wn_rows.append({"Attempt": "Levels + GARCH(1,1) on residuals", "Lag": lag, "Val RMSE": val_rmse,
                            **white_noise_summary(z, lag)})

wn_table = pd.DataFrame(wn_rows)
lb_cols = ["LB corrected p (lag+20)", f"LB corrected p (lag+{CYCLE})"]
wn_table["Passes all"] = (wn_table[lb_cols].min(axis=1) > 0.05) & (wn_table["ARCH p (variance)"] > 0.05)
display(wn_table.round(4))

# where do the biggest residuals happen?
big = var_resid.abs() > 3 * var_resid.std()
p101_switch = normal_1min["P101"].diff().ne(0)
near_switch = p101_switch.rolling(5, center=True).max().reindex(var_resid.index).astype(bool)

Each row is one way of transforming the data / one lag order. 'Val RMSE' shows whether accuracy suffered.
Passes all = strict Ljung-Box p > 0.05 at both lags AND ARCH p > 0.05 (constant variance).

Residuals larger than 3 std: 61; within 2 minutes of a P101 pump switch: 40 (66%).
Share of all minutes that are within 2 minutes of a switch: 15%.
If the first % is much bigger than the second, the leftover errors come from the pump turning on/off,
a sudden switch that a linear VAR cannot represent exactly.

In [ ]:
all_mask = np.ones(len(normal_1min), dtype=bool)
var_full, var_full_mean, var_full_std = zscore(normal_1min[var_columns], all_mask)
var_final_model = VAR(var_full).fit(var_lag)          # same columns and lag, refit on all normal data

mid, lower, upper = var_final_model.forecast_interval(var_full.to_numpy()[-var_lag:], steps=60, alpha=0.05)
to_mm = lambda z: z[:, 0] * var_full_std[var_target] + var_full_mean[var_target]   # column 0 = LIT101
var_future_index = pd.date_range(normal_1min.index[-1] + pd.Timedelta(minutes=1), periods=60, freq="min")
var_future = pd.DataFrame({"Forecast": to_mm(mid), "Lower 95%": to_mm(lower), "Upper 95%": to_mm(upper)},
                          index=var_future_index)

recent = normal_1min[var_target].iloc[-uni_plot_hours * 60:]
plt.figure(figsize=(14, 5))
plt.plot(recent.index, recent, label="Recent normal-operation data")
plt.plot(var_future.index, var_future["Forecast"], "--", label="VAR next 60 minutes")
plt.fill_between(var_future.index, var_future["Lower 95%"], var_future["Upper 95%"], alpha=0.2, label="95% interval")
plt.title("LIT101 - next 60 minutes, VAR forecast"); plt.xlabel("Time"); plt.ylabel("Water level (mm)")
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
display(var_future.head(10).round(2))

# Evaluate Performance

In [ ]:

# Every model's test predictions, keyed by timestamp
all_predictions = {
    "Persistence (previous minute)": pd.Series(
        X_test_full["lag_1"].values, index=X_test_full.index
    ),
    f"{uni_best_name} (one-step)": uni_onestep,
    f"VAR, {len(var_columns) - 1} sensors (one-step)": var_pred,
    "CatBoost (tuned)": pd.Series(y_pred_cat, index=y_test_final.index),
    "XGBoost (tuned)": pd.Series(y_pred_xgb, index=y_test_final.index),

    # Smoothing baselines
    "MA (one-step)": smoothing_test_predictions["MA"],
    "WMA (one-step)": smoothing_test_predictions["WMA"],
    "EMA (one-step)": smoothing_test_predictions["EMA"],
}

# Align all models to the same test timestamps
actual_all = ml_df[target]
common = y_test_final.index

for p in all_predictions.values():
    common = common.intersection(p.index)

# Evaluate all models
final_comparison = pd.DataFrame({
    name: regression_metrics(actual_all.loc[common], p.loc[common])
    for name, p in all_predictions.items()
}).T.sort_values("RMSE")

print(f"Scored on {len(common)} common test timestamps")
display(final_comparison.round(4))

# MAE and RMSE comparison
final_comparison[["MAE", "RMSE"]].plot.barh(figsize=(11, 6))

plt.title("Test Error Comparison: One-Step Forecasting")
plt.xlabel("Error (mm)")
plt.ylabel("Model")
plt.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.show()


# Save models

In [ ]:
from pathlib import Path
import joblib

output_dir = Path("saved_models")
output_dir.mkdir(exist_ok=True)

cat_final.save_model(str(output_dir / "catboost_tuned.cbm"))
xgb_final.save_model(str(output_dir / "xgboost_tuned.json"))
uni_best_fit.save(str(output_dir / "arima.pkl"))
joblib.dump(uni_scaler, output_dir / "arima_scaler.pkl")
joblib.dump({"model": var_model, "columns": var_columns, "mean": var_mean, "std": var_std},
            output_dir / "var.pkl")   # z-score stats are needed to turn VAR output back into mm
joblib.dump(X_train_final.columns.tolist(), output_dir / "ml_features.pkl")
print("Saved:", sorted(p.name for p in output_dir.iterdir()))

In [ ]:
# Retraining catboost
import numpy as np
import pandas as pd
import joblib

from pathlib import Path
from scipy.fft import fft, fftfreq

# Files
DATA_PATH = Path("swat_post_processed.csv")
MODEL_DIR = Path("saved_models")

# Load processed dataset
processed_clean = pd.read_csv(
    DATA_PATH,
    parse_dates=["Timestamp"],
    index_col="Timestamp"
).sort_index()

target = "LIT101"

# Reproduce the original time boundaries
WARMUP_END = pd.Timestamp("2015-12-23 15:00:00")

first_attack = (
    processed_clean.index[
        processed_clean["Normal/Attack"].eq("Attack")
    ].min().floor("min")
)

normal_minutes = pd.date_range(
    max(WARMUP_END, processed_clean.index.min().ceil("min")),
    first_attack - pd.Timedelta(minutes=1),
    freq="min"
)

development_size = int(len(normal_minutes) * 0.8)
TEST_START = normal_minutes[development_size]
VAL_START = normal_minutes[int(development_size * 0.8)]

normal_1min = (
    processed_clean.loc[
        processed_clean.index < first_attack
    ].asfreq("min")
)

assert normal_1min.index.equals(normal_minutes)
assert normal_1min["LIT101"].notna().all()
assert normal_1min["sample_count"].eq(60).all()

ml_df = normal_1min[[target]].copy()

# Reproduce FFT cycle selection from fitting data
s = ml_df.loc[ml_df.index < VAL_START, target]

frequencies = fftfreq(len(s), d=1)
magnitudes = np.abs(fft((s - s.mean()).to_numpy()))

band = (
    (frequencies > 1 / 360) &
    (frequencies < 1 / 5)
)

CYCLE = int(round(
    1 / frequencies[band][np.argmax(magnitudes[band])]
))

print("Detected cycle:", CYCLE)

# Same feature engineering as original notebook
def create_all_features(data, cycle=CYCLE):
    df = data.copy()

    df["hour"] = df.index.hour
    df["sin_hour"] = np.sin(2 * np.pi * df.index.hour / 24)
    df["cos_hour"] = np.cos(2 * np.pi * df.index.hour / 24)

    df["trend"] = (
        (df.index - ml_df.index[0]) /
        pd.Timedelta(minutes=1)
    ).to_numpy()

    df["trend_squared"] = df["trend"] ** 2

    for lag in sorted({1, 2, 3, 6, 12, 24, cycle}):
        df[f"lag_{lag}"] = df[target].shift(lag)

    for window in [3, 6, 12]:
        previous = df[target].shift(1)

        df[f"rolling_mean_{window}"] = (
            previous.rolling(window).mean()
        )
        df[f"rolling_std_{window}"] = (
            previous.rolling(window).std()
        )

    df[f"ratio_lag1_lag{cycle}"] = (
        df["lag_1"] /
        df[f"lag_{cycle}"].replace(0, np.nan)
    )

    feature_cols = df.columns.drop(target)
    df[feature_cols] = df[feature_cols].replace(
        [np.inf, -np.inf], np.nan
    )

    return df.dropna(subset=feature_cols)


features = create_all_features(ml_df)

train_feat = features.loc[features.index < TEST_START]
test_feat = features.loc[features.index >= TEST_START]

X_train_full = train_feat.drop(columns=target)
X_test_full = test_feat.drop(columns=target)

y_train = train_feat[target]
y_test_final = test_feat[target]

split = int((X_train_full.index < VAL_START).sum())

X_fit_full = X_train_full.iloc[:split]
X_val_full = X_train_full.iloc[split:]

y_train_final = y_train.iloc[:split]
y_val_final = y_train.iloc[split:]

# Load EXACT feature selection from saved notebook
selected_features = joblib.load(
    MODEL_DIR / "ml_features.pkl"
)

X_train_final = X_fit_full[selected_features]
X_val_final = X_val_full[selected_features]
X_test_final = X_test_full[selected_features]

assert X_test_final.index.equals(
    ml_df.loc[ml_df.index >= TEST_START].index
)

print("\nDataset successfully reconstructed")
print("Training:", X_train_final.shape)
print("Validation:", X_val_final.shape)
print("Test:", X_test_final.shape)
print("Features:", selected_features)
print("Training ends:", X_train_final.index[-1])
print("Validation starts:", X_val_final.index[0])
print("Test starts:", X_test_final.index[0])


In [ ]:

from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.metrics import root_mean_squared_error

cat_old = CatBoostRegressor()
cat_old.load_model(str(MODEL_DIR / "catboost_tuned.cbm"))

xgb_old = XGBRegressor()
xgb_old.load_model(str(MODEL_DIR / "xgboost_tuned.json"))

# Verify features before continuing
assert cat_old.feature_names_ == selected_features
assert xgb_old.get_booster().feature_names == selected_features

print("CatBoost trees:", cat_old.tree_count_)
print("XGBoost trees:", xgb_old.get_booster().num_boosted_rounds())

print("CatBoost validation RMSE:",
      root_mean_squared_error(
          y_val_final, cat_old.predict(X_val_final)
      ))

print("XGBoost validation RMSE:",
      root_mean_squared_error(
          y_val_final, xgb_old.predict(X_val_final)
      ))


In [ ]:

cat_extended = CatBoostRegressor(
    iterations=5000,  # Additional iterations
    depth=cat_old.get_param("depth"),
    learning_rate=cat_old.get_param("learning_rate"),
    bootstrap_type=cat_old.get_param("bootstrap_type"),
    subsample=cat_old.get_param("subsample"),
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42
)

cat_extended.fit(
    X_train_final,
    y_train_final,
    init_model=cat_old,
    eval_set=(X_val_final, y_val_final),
    early_stopping_rounds=200,
    verbose=100
)

print("Total trees:", cat_extended.tree_count_)
print("Validation RMSE:",
      root_mean_squared_error(
          y_val_final,
          cat_extended.predict(X_val_final)
      ))

cat_extended.save_model(
    str(MODEL_DIR / "catboost_extended.cbm")
)


In [ ]:

results = pd.DataFrame({
    "Model": [
        "CatBoost Original",
        "CatBoost Extended",
        "XGBoost Original"    ],
    "Validation RMSE": [
        root_mean_squared_error(
            y_val_final, m.predict(X_val_final)
        )
        for m in [
            cat_old, cat_extended,
            xgb_old
        ]
    ]
})

display(results.sort_values("Validation RMSE"))


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

models = {
    "CatBoost Original": cat_old,
    "CatBoost Extended": cat_extended,
    "XGBoost Original": xgb_old
}

test_results = []
predictions = pd.DataFrame(
    {"Actual": y_test_final},
    index=y_test_final.index
)

for name, model in models.items():
    pred = model.predict(X_test_final)
    actual = y_test_final.to_numpy()

    mae = mean_absolute_error(actual, pred)
    mse = mean_squared_error(actual, pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(actual, pred)

    nonzero = np.abs(actual) > 1e-8
    mape = np.mean(
        np.abs((actual[nonzero] - pred[nonzero]) / actual[nonzero])
    ) * 100

    test_results.append({
        "Model": name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "MAPE (%)": mape,
        "R2": r2
    })

    predictions[name] = pred

test_metrics = pd.DataFrame(test_results)

display(test_metrics.sort_values("RMSE"))
display(predictions.head(20))

In [ ]:
from catboost import CatBoostRegressor

cat_fresh = CatBoostRegressor(
    iterations=15000,
    depth=6,
    learning_rate=0.03,
    subsample=1.0,
    bootstrap_type="Bernoulli",
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42
)

cat_fresh.fit(
    X_train_final,
    y_train_final,
    eval_set=(X_val_final, y_val_final),
    early_stopping_rounds=200,
    use_best_model=True,
    verbose=500
)

print("Best iteration:", cat_fresh.best_iteration_)
print("Best validation RMSE:", cat_fresh.best_score_["validation"]["RMSE"])

cat_fresh.save_model("catboost_fresh.cbm")

In [ ]:
from sklearn.metrics import root_mean_squared_error
import pandas as pd

comparison = pd.DataFrame([
    {
        "Model": name,
        "Trees": (
            model.tree_count_
            if "CatBoost" in name
            else model.best_iteration + 1
            if hasattr(model, "best_iteration")
            else model.get_booster().num_boosted_rounds()
        ),
        "Validation RMSE": root_mean_squared_error(
            y_val_final,
            model.predict(X_val_final)
        )
    }
    for name, model in {
        "Original CatBoost": cat_old,
        "Fresh CatBoost": cat_fresh,
        "Original XGBoost": xgb_old,
    }.items()
])

display(comparison.sort_values("Validation RMSE"))

In [ ]:
cat_test_pred = cat_fresh.predict(X_test_final)
cat_test_rmse = np.sqrt(mean_squared_error(y_test_final, cat_test_pred))
print(f"CatBoost Test RMSE: {cat_test_rmse:.4f}")

